# MemTrust-Lite: quick look at the data

Synthetic agent-memory scenarios and the results of three retrieval policies. **No LLM was called**: the rates describe the simulator, not real agents.
Code and full write-up: github.com/MHKhan50/agentic-ai-research-log (folder `memtrust-agent-memory-benchmark`).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

def find(name):
    roots = [Path('/kaggle/input'), Path('.'), Path('../data'), Path('../outputs')]
    for root in roots:
        if root.exists():
            hits = [root / name] if (root / name).exists() else list(root.glob('**/' + name))
            if hits:
                return hits[0]
    raise FileNotFoundError(name)

scenarios = pd.read_csv(find('scenarios.csv'))
results = pd.read_csv(find('results.csv'))
summary = pd.read_csv(find('summary.csv'))
seeds = pd.read_csv(find('seed_summary.csv'))
print(len(scenarios), 'scenarios,', len(results), 'policy episodes')

In [ ]:
# Attack success among episodes that really contained a poisoned memory
table = summary[summary.n_poisoned_episodes > 0].pivot(
    index='poisoning_rate', columns='policy', values='attack_success_rate_on_poisoned').astype(float)
(table * 100).round(1)

In [ ]:
# Security vs utility at 25% injection
s = summary[summary.poisoning_rate == 0.25].set_index('policy')
fig, ax = plt.subplots(figsize=(7, 3.5))
x = range(len(s))
ax.bar([i - 0.2 for i in x], s.attack_success_rate_on_poisoned.astype(float) * 100, 0.4, label='attack success')
ax.bar([i + 0.2 for i in x], s.benign_task_success_rate * 100, 0.4, label='benign success')
ax.set_xticks(list(x)); ax.set_xticklabels(['relevance', 'provenance', 'prov. + corroboration'])
ax.set_ylabel('%'); ax.set_title('25% injection (synthetic)'); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
# Why corroboration scores 0%: every injected memory has exactly one source
print('corroboration of injected memories:', sorted(scenarios.loc[scenarios.poison_present == 1, 'poison_corroboration'].astype(int).unique().tolist()))
print('share of correct clean memories with one source:', round((scenarios.primary_clean_corroboration == 1).mean(), 3))

In [ ]:
# How much do the rates move between seeds?
seeds[seeds.n_poisoned_episodes > 0].pivot_table(
    index=['poisoning_rate', 'policy'], columns='seed', values='attack_success_rate_on_poisoned').astype(float).round(3)

**Reading the results.** The trust filter removes most poisoned picks with no benign cost here. The corroboration rule reaches 0% only because the simulator gives every injected memory a single source, and it loses about 8% of correct clean memories for the same reason. Next step: give the same memories to a small local LLM and count how often it follows the injected one.